# Computational performance analysis - Softmax Regression

Global imports

In [ ]:
from river import linear_model, optim, evaluate, metrics
from tabulate import tabulate
import sys
import importlib.util
import os
import time
import tracemalloc

Initial environment configuration

In [ ]:
# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [ ]:
DEFAULT_LR = 0.01
DEFAULT_L2 = 0.0
MAX_INSTANCES = 100000
SEED = 42

Dynamically load the custom SoftmaxRegression over the installed capymoa package

In [ ]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_softmax_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._softmax_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
SoftmaxRegression = module.SoftmaxRegression

Global functions

In [ ]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if i > MAX_INSTANCES:
            break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}
        x["bias"] = 1.0 # add a constant feature to simulate the bias/intercept, since River's SoftmaxRegression has no explicit bias term.

        # label
        y = instance.y_index

        data.append((x, y))

    return data

In [ ]:
from capymoa.evaluation import prequential_evaluation # We need to import capymoa's methods after setting the custom jar

def evaluateStream(stream_factory, lr=DEFAULT_LR, l2=DEFAULT_L2):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, l2)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, l2)

    table = []
    for key in ["Time (s)", "Memory (MB)"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.4f}",
            f"{river:.4f}",
            f"{(capy - river):+.4f}"
        ])

    for key in ["Accuracy", "F1"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, l2):
    softmax_reg_capymoa = SoftmaxRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        l2_penalty=l2
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using CapyMOA built-in function
    results = prequential_evaluation(
        stream=stream,
        learner=softmax_reg_capymoa,
        max_instances=MAX_INSTANCES
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = {
        "Accuracy": results['cumulative'].accuracy(),
        "F1": results['cumulative'].f1_score(),
        "Time (s)": end_time - start_time,
        "Memory (MB)": peak_memory / (1024 * 1024)
    }

    return metrics

def _evaluateStreamOnRiver(stream, lr, l2):
    softmax_reg_river = linear_model.SoftmaxRegression(
        optimizer=optim.SGD(lr),
        l2=l2
    )

    metric = (
        metrics.Accuracy() +
        metrics.F1()
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using River built-in function
    result = evaluate.progressive_val_score(
        dataset=stream,
        model=softmax_reg_river,
        metric=metric
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    results = {}
    results["Accuracy"] = float(result[0].get()*100)
    results["F1"] = float(result[1].get()*100)

    results["Time (s)"] = end_time - start_time
    results["Memory (MB)"] = peak_memory / (1024 * 1024)

    return results

## Electricity dataset (2 classes)

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)

## RandomRBFGenerator

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)

## WaveformGenerator

In [ ]:
from capymoa.stream.generator import WaveformGenerator

def make_stream():
    return WaveformGenerator(instance_random_seed=SEED)

evaluateStream(make_stream)

## RTG_2abrupt dataset

In [ ]:
from capymoa.datasets import RTG_2abrupt

evaluateStream(RTG_2abrupt)

## RandomTreeGenerator

In [ ]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=5,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)

## RandomRBFGeneratorDrift

In [ ]:
from capymoa.stream.generator import RandomRBFGeneratorDrift

def make_stream():
    return RandomRBFGeneratorDrift(
        number_of_classes=5,
        number_of_attributes=10,
        number_of_centroids=50,
        number_of_drifting_centroids=2,
        magnitude_of_change=0.5,
    )

evaluateStream(make_stream)

## RandomRBFGenerator (changed model parameters)

### L2

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream, l2=0.01)

### Learning rate

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream, lr=0.1)